# E-commerce customer behaviour

Locating the data

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/data/"))

Load the CSV file:

In [0]:
file_path = "/Volumes/workspace/default/data/Ecommerce.csv"

df = spark.read.csv(
    file_path,
    header=True,
    inferSchema=True
)

In [0]:
df.show(5)

In [0]:
df.printSchema()

In [0]:
print("Rows:", df.count())
print("Columns:", len(df.columns))

In [0]:
print(df.columns)

In [0]:
df.describe().show()

In [0]:
df.show(10, truncate=False)

check missing values:

In [0]:
from pyspark.sql.functions import col, sum

missing_values = df.select(
    [
        sum(col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ]
)

missing_values.show()

check duplicates in rows and columns:

In [0]:
total_rows = df.count()
unique_rows = df.dropDuplicates().count()

print("Total rows:", total_rows)
print("Unique rows:", unique_rows)
print("Duplicate rows:", total_rows - unique_rows)

## Business question 1: 

What percentage of website sessions result in a purchase, and what does the customer journey look like?

If we look for a target variable, we see that ```purchased``` is one. Because it contains 2 classes. So this can be a binary classification problem.

Let's see the distribution of target variable:

In [0]:
df.groupBy("purchased").count().orderBy("purchased").show()

We see the data is a little imbalanced. Now lets calculate the purchased rate:

In [0]:
from pyspark.sql.functions import avg

purchase_rate = df.select(
    (avg("purchased") * 100).alias("purchase_rate_percent")
)

purchase_rate.show()

## Business question 2:

Analyze the shopping funnel <br>

Now let's calculate:<br>

- Total sessions
- Sessions where customers added to cart
- Sessions where customers purchased
- Sessions where customers abandoned their cart

In [0]:
from pyspark.sql.functions import sum, col

funnel = df.select(
    sum(col("added_to_cart")).alias("added_to_cart"),
    sum(col("purchased")).alias("purchased"),
    sum(col("cart_abandoned")).alias("cart_abandoned")
)

funnel.show()

calculating percentage from the counts, as they are more useful:

In [0]:
total_sessions = df.count()

funnel_percent = df.select(
    (sum("added_to_cart") / total_sessions * 100).alias("cart_add_rate"),
    (sum("purchased") / total_sessions * 100).alias("purchase_rate"),
    (sum("cart_abandoned") / total_sessions * 100).alias("cart_abandonment_rate")
)

funnel_percent.show()

## Business question 3:

Does adding something to the cart strongly relate to purchasing?

In [0]:
from pyspark.sql.functions import count, avg

df.groupBy("added_to_cart").agg(
    count("*").alias("sessions"),
    (avg("purchased") * 100).alias("purchase_rate")
).orderBy("added_to_cart").show()

## Analyze customer engagement

Now let's investigate whether people who spend more time on the website are more likely to purchase. It will be interesting to ask How does customer engagement differ between purchasers and non-purchasers?

In [0]:
df.groupBy("purchased").agg(
    avg("pages_viewed").alias("avg_pages_viewed"),
    avg("time_on_site_sec").alias("avg_time_on_site_sec")
).orderBy("purchased").show()

## Analyze marketing channels

Let's see whether some marketing channels have higher conversion.

We are asking: Which marketing channels generate the most purchasing behavior?

And:

Do channels with higher conversion also generate higher revenue?

In [0]:
df.groupBy("marketing_channel").agg(
    count("*").alias("sessions"),
    (avg("purchased") * 100).alias("purchase_rate"),
    avg("revenue").alias("avg_revenue")
).orderBy(col("purchase_rate").desc()).show()

## Analyze product categories

lets compare product categories:

In [0]:
df.groupBy("product_category").agg(
    count("*").alias("sessions"),
    (avg("purchased") * 100).alias("purchase_rate"),
    avg("unit_price").alias("avg_product_price"),
    sum("revenue").alias("total_revenue")
).orderBy(col("total_revenue").desc()).show()

Let's inspect the distributions first.

In [0]:
df.groupBy("device_type").count().orderBy("device_type").show()

In [0]:
df.groupBy("user_type").count().orderBy("user_type").show()

In [0]:
df.groupBy("payment_method").count().orderBy("payment_method").show()

In [0]:
df.groupBy("payment_method").count().orderBy("payment_method").show()

In [0]:
df.groupBy("visit_season").count().orderBy("visit_season").show()

## One genuinely useful PySpark analysis
Does the conversion rate differ depending on device type?

In [0]:
df.groupBy("device_type").agg(
    count("*").alias("sessions"),
    (avg("purchased") * 100).alias("purchase_rate"),
    avg("time_on_site_sec").alias("avg_time_on_site")
).orderBy(col("purchase_rate").desc()).show()

## Purchase Prediction
Can we predict whether an e-commerce session will result in a purchase based on the customer's behavior? SO ```Purchase```is our target variable. Let's start with a simple Spark ML Logistic Regression model.

In [0]:
df.printSchema()

Out of the given columns, let's use the below columns: 
- device_type
- user_type
- marketing_channel
- product_category
- unit_price
- quantity
- discount_percent
- pages_viewed
- time_on_site_sec
- added_to_cart

However, there is an important ML issue here. Some columns describe things that happen after or essentially at the moment of purchase. So these can lead to Data Leakage. Columns like: 
- revenue
- discount_amount
- cart_abandoned

shouldn't be used for our first prediction model. Otherwise this could cheat our model.

## Create our ML DataFrame

In [0]:
feature_columns = [
    "device_type",
    "user_type",
    "marketing_channel",
    "product_category",
    "unit_price",
    "quantity",
    "discount_percent",
    "pages_viewed",
    "time_on_site_sec",
    "added_to_cart"
]

ml_df = df.select(
    feature_columns + ["purchased"]
)

ml_df.show(5)

In [0]:
ml_df.printSchema()

In [0]:
print("Rows:", ml_df.count())
print("Columns:", len(ml_df.columns))

## Do one final ML data-quality check

In [0]:
ml_df.groupBy("purchased").count().orderBy("purchased").show()

So we see that our target is somewhat imbalanced:

- ~77.5% no purchase
- ~22.5% purchase

That's not extreme, but it means accuracy alone won't be enough to evaluate the model. So we will be looking at other evaluation metric values like:

- precision
- recall
- F1 score
- AUC

## Save the processed data as a Delta table

This is where we start getting hands-on with Databricks + Delta Lake. Instead of continually working with the original CSV, we'll save our prepared dataset in Delta format.

In [0]:
ml_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_ml_data")

In [0]:
spark.table("ecommerce_ml_data").show(5)

In [0]:
spark.table("ecommerce_ml_data").count()

We actually loaded the raw CSV into Databricks using PySpark, transformed it into a machine-learning dataset, and stored the processed data as a Delta table.

## transformations vs actions

Let's experience the Lazy evaluation happening here:

In [0]:
filtered_df = ml_df.filter(
    col("pages_viewed") > 10
)

At this point, Spark has not necessarily executed the computation. We have created a transformation/plan.

In [0]:
filtered_df.count()

```count()``` is an action, so Spark actually needs to execute the computation. Transformations are lazy; actions trigger execution.

## Split the data into train and test

We'll use: 80% training / 20% testing

In [0]:
train_df, test_df = ml_df.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Training rows:", train_df.count())
print("Test rows:", test_df.count())

## Build our first Spark ML pipeline

Some of the features are catagorical. Spark ML needs us to transform those into numerical feature representations.

In [0]:
#import tools
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression

Identify catagorical and numerical columns:

In [0]:
categorical_columns = [
    "device_type",
    "user_type",
    "marketing_channel",
    "product_category"
]

numerical_columns = [
    "unit_price",
    "quantity",
    "discount_percent",
    "pages_viewed",
    "time_on_site_sec",
    "added_to_cart"
]

Our problem is to predict whether a session will result in a purchase (purchased = 1).

We have 2 types of features: Categorical:

- device_type
- user_type
- marketing_channel
- product_category

And Numerical:

- unit_price
- quantity
- discount_percent
- pages_viewed
- time_on_site_sec
- added_to_cart

In [0]:
# Create indexers for categorical variables
indexers = [
    StringIndexer(
        inputCol=column,
        outputCol=column + "_index",
        handleInvalid="keep"
    )
    for column in categorical_columns
]

```StringIndexer``` converts categorical values into numerical indices that Spark ML can work with.

Even though your categories are already represented by numbers (0, 1, 2, etc.), we still treat them as categories rather than continuous numerical quantities.

In [0]:
# One-hot encode them
encoder = OneHotEncoder(
    inputCols=[column + "_index" for column in categorical_columns],
    outputCols=[column + "_encoded" for column in categorical_columns]
)

Combine all features: Now we need to put everything into one feature vector.

In [0]:
assembler = VectorAssembler(
    inputCols=[
        column + "_encoded" for column in categorical_columns
    ] + numerical_columns,
    outputCol="features"
)

## Create the Logistic Regression model:

In [0]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol="purchased"
)

## Put everything into one Pipeline

In [0]:
pipeline = Pipeline(
    stages=indexers + [encoder, assembler, lr]
)

Above is an important Spark ML concept. Instead of manually doing:
1. transform categorical columns
2. encode columns
3. assemble features
4. train model

we define a Pipeline that handles these stages consistently. That's very useful in real ML workflows because the exact same preprocessing can be applied during training and prediction.

## Train the model

In [0]:
model = pipeline.fit(train_df)
print("Model training completed.")

## Make predictions

In [0]:
predictions = model.transform(test_df)

predictions.select(
    "purchased",
    "prediction",
    "probability"
).show(10, truncate=False)

Here,
- ```purchased``` → actual result
- ```prediction``` → model's predicted class
- ```probability``` → model's estimated probability

## Evaluate the model

This is important because we don't just want to train a model—we want to know whether it works.

In [0]:
#calculate accuracy
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="purchased",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = accuracy_evaluator.evaluate(predictions)

print("Accuracy:", accuracy)

We see that a model that simply predicted "no purchase" for everyone would already achieve about 77.5% accuracy. That's why we're going to evaluate more than accuracy.

## Calculate precision, recall and F1

In [0]:
# Calculate weighted evaluation metrics and save them separately

weighted_precision = None
weighted_recall = None
weighted_f1 = None

for metric_name in ["weightedPrecision", "weightedRecall", "f1"]:
    evaluator = MulticlassClassificationEvaluator(
        labelCol="purchased",
        predictionCol="prediction",
        metricName=metric_name
    )

    metric_value = evaluator.evaluate(predictions)

    if metric_name == "weightedPrecision":
        weighted_precision = metric_value
    elif metric_name == "weightedRecall":
        weighted_recall = metric_value
    elif metric_name == "f1":
        weighted_f1 = metric_value

print("Weighted Precision:", weighted_precision)
print("Weighted Recall:", weighted_recall)
print("Weighted F1:", weighted_f1)

In [0]:
#Check your AUC
from pyspark.ml.evaluation import BinaryClassificationEvaluator

auc_evaluator = BinaryClassificationEvaluator(
    labelCol="purchased",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = auc_evaluator.evaluate(predictions)

print("AUC:", auc)

For this project below metrices:
- Precision: Of the sessions we predicted would purchase, how many actually purchased?
- Recall: Of all the sessions that actually purchased, how many did we identify?
- F1: A balance between precision and recall.

makes more sense than accuracy

## Experimental tracking using MLFlow

In [0]:
#check mlflow is available
import mlflow

print("MLflow version:", mlflow.__version__)

MLflow should handle the temporary storage.

In [0]:
import os

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/data/mlflow_tmp"

print("MLFLOW_DFS_TMP:", os.environ["MLFLOW_DFS_TMP"])

In [0]:
import mlflow
import mlflow.spark

with mlflow.start_run():

    # Log model parameters
    mlflow.log_param("model", "Logistic Regression")
    mlflow.log_param("train_test_split", "80/20")
    mlflow.log_param("random_seed", 42)

    # Log evaluation metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("weighted_precision", weighted_precision)
    mlflow.log_metric("weighted_recall", weighted_recall)
    mlflow.log_metric("weighted_f1", weighted_f1)
    mlflow.log_metric("auc", auc)

    # Log the trained Spark ML pipeline
    mlflow.spark.log_model(
        model,
        "logistic_regression_model"
    )

    print("MLflow run completed and model logged.")

What we did is we are telling MLflow the parameters and metrices and MLflow stores these together as one experiment run.

In [0]:
import mlflow

run_id = "f65015fca6a64036bdcd5bd1b96b0931"

model_uri = f"runs:/{run_id}/logistic_regression_model"

print(model_uri)

My model uri: runs:/f65015fca6a64036bdcd5bd1b96b0931/logistic_regression_model. Now lets load this model from MLflow.

In [0]:
import mlflow

model_uri = "runs:/f65015fca6a64036bdcd5bd1b96b0931/logistic_regression_model"

loaded_model = mlflow.spark.load_model(model_uri)

print("Model loaded successfully!")

## Make predictions with the loaded model

In [0]:
loaded_predictions = loaded_model.transform(test_df)

print("Predictions created successfully!")

## Look at the predictions

In [0]:
loaded_predictions.select(
    "purchased",
    "prediction",
    "probability"
).show(10)

- purchased	: What actually happened
- prediction : What our model predicted
- probability :	How confident the model was

## Now let's make sure the loaded model gives the same accuracy as the original model.

In [0]:
accuracy_loaded = accuracy_evaluator.evaluate(loaded_predictions)

print("Accuracy from loaded model:", accuracy_loaded)

Now lets' include Gen AI to this project: We'll use your review_text data and build a simple review analysis / semantic search component so that you can honestly say you experimented with embeddings, vector search, and an LLM in Databricks.

Let's start by inspecting what review_text contains

In [0]:
df.select("review_text").show(20, truncate=False)

In [0]:
df.select("review_text").distinct().show(20, truncate=False)

In [0]:
#print original data
print(df)

Let's use LLM where we use the categorical information + numerical metrics to create a customer/session profile, then ask an LLM to interpret it.

For example, we can create a structured record like:
- Customer session:
- Session duration: Very Long
- Pages viewed: 19
- Time on site: 1674 seconds
- Added to cart: Yes
- Purchased: Yes
- Product category: 3
- Quantity: 2
- Discount: 5%
- Revenue: 2410.23
- Rating: 4

Then an LLM can turn that into something like:

"This session shows strong engagement, with a long visit and multiple pages viewed. The customer added an item to the cart and completed a purchase, generating relatively high revenue."

So this project will be something like:
PySpark + Databricks → EDA → Spark ML → MLflow → LLM-powered business insights

In [0]:
# let's understand the four session-duration categories.
from pyspark.sql.functions import count, avg, round
df.groupBy("session_duration_bucket").agg(
    count("*").alias("sessions"),
    round(avg("purchased") * 100, 2).alias("purchase_rate_percent"),
    round(avg("pages_viewed"), 2).alias("avg_pages"),
    round(avg("time_on_site_sec"), 2).alias("avg_time_sec"),
    round(avg("revenue"), 2).alias("avg_revenue")
).orderBy("purchase_rate_percent", ascending=False).show()

The main observations are:

- Very Short: 20.24% purchase rate
- Short: 22.79%
- Very Long: 23.22%
- Long: 23.62% — highest

So there is a small positive relationship between longer sessions and purchasing, although we should not claim that longer sessions cause purchases. Also interesting:

- Very Short sessions average only 230.78 seconds and have the lowest average revenue (345.85).
- Long sessions average 1,131.75 seconds and have the highest average revenue (435.06).
- Very Long sessions spend the most time (1,576.27 sec) but don't have the highest revenue.

Now we can create a structured business profile from these findings and let an LLM turn the numbers into a human-readable insight. For example, instead of feeding the LLM raw rows, we could give it:

Session Duration Analysis

Very Short:
- Sessions: 6,270
- Purchase rate: 20.24%
- Average pages: 12.53
- Average time: 230.78 sec
- Average revenue: 345.85

Short:
- Sessions: 6,236
- Purchase rate: 22.79%
- Average pages: 12.44
- Average time: 676.83 sec
- Average revenue: 418.52

Long:
- Sessions: 6,254
- Purchase rate: 23.62%
- Average pages: 12.48
- Average time: 1,131.75 sec
- Average revenue: 435.06

Very Long:
- Sessions: 6,240
- Purchase rate: 23.22%
- Average pages: 12.69
- Average time: 1,576.27 sec
- Average revenue: 419.38

Then the LLM could generate something like:

"Long sessions show the highest purchase rate and average revenue, while very short sessions have the lowest conversion and revenue. This suggests that extremely short visits may represent lower-intent sessions. However, the difference between long and very long sessions indicates that simply spending more time on the website does not necessarily lead to higher revenue."

## let's build the structured insight data

In [0]:
# Before touching an LLM, let's make one small PySpark table that contains these business metrics.
from pyspark.sql.functions import count, avg, round

session_insights = df.groupBy(
    "session_duration_bucket"
).agg(
    count("*").alias("sessions"),
    round(avg("purchased") * 100, 2).alias("purchase_rate_percent"),
    round(avg("pages_viewed"), 2).alias("avg_pages"),
    round(avg("time_on_site_sec"), 2).alias("avg_time_sec"),
    round(avg("revenue"), 2).alias("avg_revenue")
)

session_insights.show()

So we see here that Very short sessions have noticeably lower conversion and revenue. Longer sessions generally perform better, but extremely long sessions don't provide additional revenue compared with long sessions. That's a nice insight for our LLM because it has something meaningful to explain rather than just asking an LLM to summarize raw data..

In [0]:
session_data = [
    row.asDict()
    for row in session_insights.collect()
]

session_data

In [0]:
# create a text representation for the LLM
#Let's keep this very simple. We will take the session_insights Spark DataFrame and convert it into a Python list of #dictionaries.
session_data = [
    {'session_duration_bucket': 'Short',
     'sessions': 6236,
     'purchase_rate_percent': 22.79,
     'avg_pages': 12.44,
     'avg_time_sec': 676.83,
     'avg_revenue': 418.52},

    {'session_duration_bucket': 'Very Short',
     'sessions': 6270,
     'purchase_rate_percent': 20.24,
     'avg_pages': 12.53,
     'avg_time_sec': 230.78,
     'avg_revenue': 345.85},

    {'session_duration_bucket': 'Long',
     'sessions': 6254,
     'purchase_rate_percent': 23.62,
     'avg_pages': 12.48,
     'avg_time_sec': 1131.75,
     'avg_revenue': 435.06},

    {'session_duration_bucket': 'Very Long',
     'sessions': 6240,
     'purchase_rate_percent': 23.22,
     'avg_pages': 12.69,
     'avg_time_sec': 1576.27,
     'avg_revenue': 419.38}
]

Unfortunately, as you can see there is no proper useful column in this data to be used inside LLM. SO let's stop this project here without adding LLM part!

## Project summary cell:

In [0]:
# ============================================================
# PROJECT SUMMARY
# E-Commerce Customer Intelligence with PySpark, Databricks & GenAI
# ============================================================

print("=" * 70)
print("E-COMMERCE CUSTOMER INTELLIGENCE PROJECT")
print("=" * 70)

print("\nDataset")
print("-" * 70)
print(f"Total sessions analyzed: {df.count():,}")
print(f"Purchase rate: {df.selectExpr('avg(purchased) * 100').first()[0]:.2f}%")

print("\nKey Business Insights")
print("-" * 70)
print("1. 64.47% of sessions resulted in an item being added to the cart.")
print("2. 34.85% of sessions with an added cart resulted in a purchase.")
print("3. Very short sessions had the lowest purchase rate (20.24%).")
print("4. Long sessions had the highest purchase rate (23.62%).")
print("5. Product category 6 had the highest purchase rate (24.68%).")
print("6. Product category 2 generated the highest total revenue.")

print("\nMachine Learning")
print("-" * 70)
print("Model: Logistic Regression")
print("Framework: PySpark ML")
print("Train/Test split: 80/20")
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test AUC: {auc:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")

print("\nExperiment Tracking")
print("-" * 70)
print("MLflow: Model, parameters and evaluation metrics logged")
print("Model successfully reloaded from MLflow")
print("Reloaded model reproduced the original test accuracy")

print("\nGenAI Component")
print("-" * 70)
print("Created an LLM prompt to convert structured session")
print("analytics into concise business insights and recommendations.")
print("The prompt was designed with business-focused guardrails.")
print("LLM execution was not available in the Databricks Free Edition workspace.")

print("\nProject Pipeline")
print("-" * 70)
print("Raw Data")
print("    ↓")
print("PySpark Data Processing & EDA")
print("    ↓")
print("Feature Engineering")
print("    ↓")
print("Spark ML Pipeline")
print("    ↓")
print("Logistic Regression")
print("    ↓")
print("Model Evaluation")
print("    ↓")
print("MLflow Experiment Tracking")

print("\n" + "=" * 70)
print("PROJECT COMPLETED")
print("=" * 70)